In [1]:
import pandas as pd
import numpy as np

daily = pd.read_csv("daily_wide_revenue.csv", parse_dates=["date"])
weekly = pd.read_csv("weekly_wide_revenue.csv", parse_dates=["week_start"])

print("Daily shape:", daily.shape)
print("Weekly shape:", weekly.shape)

display(daily.head())
display(weekly.head())

Daily shape: (180, 4)
Weekly shape: (26, 4)


,date,North,South,West
0,2025-01-01,50651.59,48937.65,60109.02
1,2025-01-02,54431.06,52101.37,62082.51
2,2025-01-03,62516.51,53274.36,57028.24
3,2025-01-04,67241.10,60980.94,65800.24
4,2025-01-05,66825.20,66908.57,69358.06


,week_start,North,South,West
0,2024-12-30,301665.46,282202.89,314378.07
1,2025-01-06,462640.35,403753.49,470245.55
2,2025-01-13,418070.39,373768.18,456558.55
3,2025-01-20,375315.65,372906.22,402581.03
4,2025-01-27,426816.89,373839.27,422498.49


In [2]:
regions = ["North", "South", "West"]

checks = {
    "Daily row count is 180": len(daily) == 180,
    "Weekly row count is 26": len(weekly) == 26,
    "Daily columns correct": list(daily.columns) == ["date"] + regions,
    "Weekly columns correct": list(weekly.columns) == ["week_start"] + regions,
    "Daily dates unique": not daily["date"].duplicated().any(),
    "Weekly dates unique": not weekly["week_start"].duplicated().any(),
    "Daily dates sorted": daily["date"].is_monotonic_increasing,
    "Weekly dates sorted": weekly["week_start"].is_monotonic_increasing,
    "Daily missing values absent": daily.isnull().sum().sum() == 0,
    "Weekly missing values absent": weekly.isnull().sum().sum() == 0,
    "Daily revenue numeric": daily[regions].select_dtypes(include="number").shape[1] == 3,
    "Weekly revenue numeric": weekly[regions].select_dtypes(include="number").shape[1] == 3,
}

for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} - {check}")

print("\nTotal checks:", len(checks))
print("Passed:", sum(checks.values()))
print("Failed:", len(checks) - sum(checks.values()))

PASS - Daily row count is 180
PASS - Weekly row count is 26
PASS - Daily columns correct
PASS - Weekly columns correct
PASS - Daily dates unique
PASS - Weekly dates unique
PASS - Daily dates sorted
PASS - Weekly dates sorted
PASS - Daily missing values absent
PASS - Weekly missing values absent
PASS - Daily revenue numeric
PASS - Weekly revenue numeric

Total checks: 12
Passed: 12
Failed: 0


In [3]:
daily["week_start"] = (
    daily["date"]
    - pd.to_timedelta(daily["date"].dt.dayofweek, unit="D")
)

daily_weekly_check = (
    daily.groupby("week_start")[regions]
    .sum()
    .reset_index()
)

comparison = daily_weekly_check.merge(
    weekly,
    on="week_start",
    how="outer",
    suffixes=("_daily", "_weekly"),
    indicator=True
)

print("All weekly periods matched:", (comparison["_merge"] == "both").all())

for region in regions:
    difference = (
        comparison[f"{region}_daily"] - comparison[f"{region}_weekly"]
    ).abs()

    print(f"{region} - Maximum absolute difference: {difference.max():.6f}")
    print(f"{region} - Totals match:", np.allclose(
        comparison[f"{region}_daily"],
        comparison[f"{region}_weekly"],
        rtol=1e-9,
        atol=0.01
    ))

All weekly periods matched: True
North - Maximum absolute difference: 0.000000
North - Totals match: True
South - Maximum absolute difference: 0.000000
South - Totals match: True
West - Maximum absolute difference: 0.000000
West - Totals match: True


In [4]:
print("DAY 3 - MODEL-READY DATA VALIDATION")
print("-" * 45)

print("Daily rows:", len(daily))
print("Weekly rows:", len(weekly))

print("Daily missing values:", daily.isnull().sum().sum())
print("Weekly missing values:", weekly.isnull().sum().sum())

print("Daily duplicate dates:", daily["date"].duplicated().sum())
print("Weekly duplicate dates:", weekly["week_start"].duplicated().sum())

print("\nRevenue consistency:")
for region in regions:
    daily_total = daily[region].sum()
    weekly_total = weekly[region].sum()

    print(f"{region} daily total: {daily_total:.2f}")
    print(f"{region} weekly total: {weekly_total:.2f}")
    print(f"{region} totals match: {np.isclose(daily_total, weekly_total)}")

print("\nValidation checks passed:",
      sum(checks.values()), "/", len(checks))

DAY 3 - MODEL-READY DATA VALIDATION
---------------------------------------------
Daily rows: 180
Weekly rows: 26
Daily missing values: 0
Weekly missing values: 0
Daily duplicate dates: 0
Weekly duplicate dates: 0

Revenue consistency:
North daily total: 11415313.37
North weekly total: 11415313.37
North totals match: True
South daily total: 10220580.66
South weekly total: 10220580.66
South totals match: True
West daily total: 12162190.59
West weekly total: 12162190.59
West totals match: True

Validation checks passed: 12 / 12
